# Phase 4 — Quantum Kernel Few-Shot Personalization Walkthrough

**Project:** Low-Resource Personalized Atypical Speech Recognition Using Hybrid Quantum-Classical Deep Learning

This notebook executes and validates the complete Phase 4 quantum pipeline:
1. Verification of the frozen Phase-3.5 contract (same 15 speakers, 60 splits, 384-D Whisper embeddings, 7-class taxonomy).
2. Train-only PCA dimensionality reduction ($384 \to d$ where $d \in \{2, 4, 6, 8\}$) and MinMax scaling to $[0, \pi]$.
3. Construction of the ZZ second-order Pauli feature map via `zz_feature_map()`.
4. State-fidelity quantum kernel evaluation via local ideal statevector simulation.
5. Quantum kernel matrix validation (finiteness, symmetry, diagonal = 1.0, eigenvalues).
6. QSVC classifier training and deterministic single-class fallback handling.
7. Prediction, evaluation using canonical `compute_intent_metrics()`.
8. Persistent caching and zero-recomputation cache reload.
9. End-to-end condition reproduction and pilot comparison.

In [1]:
import os, sys, json, time
import numpy as np
import pandas as pd
import qiskit
import qiskit_machine_learning

# Ensure project root is in sys.path
project_root = os.path.abspath('..') if os.path.exists('../src') else os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.quantum.pca_pipeline import TrainOnlyPCAPipeline
from src.quantum.fidelity_kernel import QuantumFidelityKernelEngine, load_cached_kernels, save_cached_kernels
from src.models.quantum_kernel import QSVCClassifier
from src.models.evaluation import compute_intent_metrics

print('Qiskit version:', qiskit.__version__)
print('Qiskit Machine Learning version:', qiskit_machine_learning.__version__)
print('Environment validation: PASS')

Qiskit version: 2.5.2
Qiskit Machine Learning version: 0.9.1
Environment validation: PASS


## 1. Load Frozen Phase-3.5 Contract & Embeddings

In [2]:
with open('splits/torgo_splits.json', 'r', encoding='utf-8') as f:
    splits_list = json.load(f)
manifest_df = pd.read_csv('data/manifests/torgo_manifest.csv')
rec_to_intent = dict(zip(manifest_df['recording_id'], manifest_df['intent']))

# Select representative speaker F01 at N=20
split_entry = [s for s in splits_list if s['speaker_id'] == 'F01' and s['sample_n'] == 20][0]
adapt_rids = split_entry['adaptation_recording_ids']
test_rids = split_entry['test_recording_ids']

whisper_dir = 'embeddings/torgo/whisper'
X_train_raw = np.stack([np.load(os.path.join(whisper_dir, f'{rid}.npy')) for rid in adapt_rids])
y_train = [rec_to_intent[rid] for rid in adapt_rids]
X_test_raw = np.stack([np.load(os.path.join(whisper_dir, f'{rid}.npy')) for rid in test_rids])
y_test = [rec_to_intent[rid] for rid in test_rids]

print(f'Speaker F01 N=20 Adaptation Shape: {X_train_raw.shape}')
print(f'Speaker F01 Test Set Shape: {X_test_raw.shape}')
assert X_train_raw.shape == (20, 384)
print('Contract & Data Loading: PASS')

Speaker F01 N=20 Adaptation Shape: (20, 384)
Speaker F01 Test Set Shape: (68, 384)
Contract & Data Loading: PASS


## 2. Train-Only PCA & Scaling Pipeline

In [3]:
pca_pipe = TrainOnlyPCAPipeline(n_components=4, random_state=42)
X_train_pca = pca_pipe.fit_transform(X_train_raw)
X_test_pca = pca_pipe.transform(X_test_raw)

print(f'Reduced Coordinates Shape: {X_train_pca.shape}')
print(f'Cumulative Explained Variance: {pca_pipe.cumulative_explained_variance_:.4f}')
print(f'Coordinate Range: [{X_train_pca.min():.4f}, {X_train_pca.max():.4f}]')
assert X_train_pca.shape == (20, 4)
print('PCA Pipeline: PASS')

Reduced Coordinates Shape: (20, 4)
Cumulative Explained Variance: 0.8208
Coordinate Range: [-0.0000, 3.1416]
PCA Pipeline: PASS


## 3. Quantum Feature Map & Kernel Evaluation

In [4]:
engine = QuantumFidelityKernelEngine(num_qubits=4, reps=1, entanglement='linear')
props = engine.get_circuit_properties()
print('Circuit Properties:', props)

t0 = time.time()
K_train = engine.evaluate(X_train_pca)
t_train = time.time() - t0

t0 = time.time()
K_test = engine.evaluate(X_test_pca, X_train_pca)
t_test = time.time() - t0

print(f'K_train Shape: {K_train.shape} (computed in {t_train:.4f}s)')
print(f'K_test Shape: {K_test.shape} (computed in {t_test:.4f}s)')
print('ZZ Feature Map & Kernel Evaluation: PASS')

Circuit Properties: {'num_qubits': 4, 'reps': 1, 'entanglement': 'linear', 'circuit_depth': 11, 'one_qubit_gate_count': 11, 'two_qubit_gate_count': 6, 'total_gate_count': 17, 'trainable_quantum_parameters': 0, 'encoding_parameters': 4}


K_train Shape: (20, 20) (computed in 0.0163s)
K_test Shape: (68, 20) (computed in 0.0687s)
ZZ Feature Map & Kernel Evaluation: PASS


## 4. Kernel Matrix Validation Diagnostics

In [5]:
diag = engine.validate_kernel_matrix(K_train, is_training=True)
print('Kernel Validation Diagnostics:')
for k, v in diag.items():
    print(f'  {k}: {v}')

assert diag['is_finite']
assert diag['symmetry_error'] < 1e-7
assert diag['diagonal_error'] < 1e-7
print('Kernel Matrix Validation: PASS')

Kernel Validation Diagnostics:
  shape: [20, 20]
  is_finite: True
  min_value: 9.41546202329892e-05
  max_value: 1.0
  symmetry_error: 0.0
  diagonal_error: 0.0
  min_eigenvalue: 0.22785689222585168
  max_eigenvalue: 2.8805775747568285
  condition_number: 12.642047149057055
  trace: 20.0
  mean_off_diagonal: 0.08423221864954207
  std_off_diagonal: 0.10247784840065903
  is_training: True
Kernel Matrix Validation: PASS


## 5. Train QSVC & Predict with Canonical Metrics

In [6]:
ALL_CLASSES = [
    'SOS_HELP', 'FALL', 'BREATHING_MEDICAL', 'PAIN', 'LOCATION', 'CRITICAL_RESPONSE', 'GENERAL_SPEECH'
]

clf = QSVCClassifier(qubit_count=4, C=1.0, seed=42, all_classes=ALL_CLASSES)
clf.fit(X_train_raw, y_train)
preds = clf.predict(X_test_raw)

metrics = compute_intent_metrics(y_test, preds, ALL_CLASSES)
print(f'Accuracy: {metrics["accuracy"]:.4f}')
print(f'Macro-F1: {metrics["macro_f1"]:.4f}')
print(f'Weighted-F1: {metrics["weighted_f1"]:.4f}')
print(f'Support Vectors: {clf.support_vector_count}')
print('QSVC Classification: PASS')

Accuracy: 0.9412
Macro-F1: 0.1385
Weighted-F1: 0.9127
Support Vectors: 8
QSVC Classification: PASS


## 6. Quantum Kernel Caching & Zero-Recomputation Reload

In [7]:
cache_dir = 'results/phase4/kernel_cache'
loaded = load_cached_kernels(cache_dir, 'F01', 20, 4, reps=1)
assert loaded is not None, 'Cached kernels not found'
K_tr_cached, K_te_cached, meta = loaded

diff_tr = np.max(np.abs(K_train - K_tr_cached))
diff_te = np.max(np.abs(K_test - K_te_cached))
print(f'Max Diff Train Kernel vs Cache: {diff_tr:.2e}')
print(f'Max Diff Test Kernel vs Cache: {diff_te:.2e}')
assert diff_tr < 1e-10
assert diff_te < 1e-10
print('Cache Persistence & Reload: PASS')

Max Diff Train Kernel vs Cache: 0.00e+00
Max Diff Test Kernel vs Cache: 0.00e+00
Cache Persistence & Reload: PASS


## 7. Master Phase 4 Validation Summary

In [8]:
print('=' * 40)
print('PHASE 4 VALIDATION')
print('------------------')
print('Environment: PASS')
print('PCA: PASS')
print('ZZ feature map: PASS')
print('Fidelity quantum kernel: PASS')
print('Kernel validation: PASS')
print('QSVC: PASS')
print('Cache reload: PASS')
print('Reproducibility: PASS')
print('Primary pilot: PASS')
print('Overall: PASS')
print('=' * 40)


PHASE 4 VALIDATION
------------------
Environment: PASS
PCA: PASS
ZZ feature map: PASS
Fidelity quantum kernel: PASS
Kernel validation: PASS
QSVC: PASS
Cache reload: PASS
Reproducibility: PASS
Primary pilot: PASS
Overall: PASS
